# Fills by trade type
Reads `out/fills.csv` (run `python3 main.py` first) and shows the top 20 rows per venue.

In [ ]:
import pandas as pd

pd.set_option("display.width", 160)
fills = pd.read_csv("out/fills.csv")
print(fills.shape)
fills.groupby(["capacity", "venue"])["quantity"].agg(fills="count", shares="sum")

In [ ]:
# INTERNAL — principal fills (firm is the counterparty)
fills[fills["venue"] == "INTERNAL"].head(20)

In [ ]:
# CROSS — client-to-client agency fills (two rows per cross, same price/time)
fills[fills["venue"] == "CROSS"].head(20)

In [ ]:
# MARKET — routed out at the touch (agency)
fills[fills["venue"] == "MARKET"].head(20)

## Unfilled orders — cancelled IOC and expired DAY
Both classes of unfilled orders, with the NBBO and the firm's principal position at arrival. IOC: cancelled immediately (non-marketable on arrival). DAY: rested all day and expired at the 16:00 close (still non-marketable at the final quote — routing them would violate their limit).

In [ ]:
import config
from internalizer.data import load_orders, load_quotes

quotes = load_quotes(config.QUOTES_CSV)
orders = load_orders(config.ORDERS_CSV)
firm = pd.read_csv(config.FIRM_TRADES_CSV, parse_dates=["timestamp"])

# quote tape as a frame (prices back to dollars)
qdf = pd.DataFrame({"ts": [q.ts for q in quotes],
                    "bid": [q.bid / 100 for q in quotes],
                    "ask": [q.ask / 100 for q in quotes]})
last_q = qdf.iloc[-1]   # prevailing NBBO at the close

filled = fills.groupby("order_id")["quantity"].sum()

# principal position timeline: principal fills (client buy = firm sells) + firm hedges
pf = fills[fills["capacity"] == "PRINCIPAL"].copy()
pf["ts"] = pd.to_datetime(pf["timestamp"])
pf["dpos"] = pf["quantity"].where(pf["side"] == "SELL", -pf["quantity"])
fh = firm.rename(columns={"timestamp": "ts"})[["ts", "side", "quantity"]].copy()
fh["dpos"] = fh["quantity"].where(fh["side"] == "BUY", -fh["quantity"])
tl = pd.concat([pf[["ts", "dpos"]], fh[["ts", "dpos"]]]).sort_values("ts")
tl["pos"] = tl["dpos"].cumsum()


def unfilled_table(tif):
    """Orders of the given TIF that did not fully fill, with arrival NBBO and position."""
    rows = []
    for o in orders:
        if o.tif != tif or filled.get(o.order_id, 0) >= o.quantity:
            continue
        q = qdf[qdf["ts"] <= o.ts].iloc[-1]              # prevailing NBBO at arrival
        p = tl[tl["ts"] <= o.ts]
        pos = int(p["pos"].iloc[-1]) if len(p) else 0
        limit = o.limit / 100
        # distance from the touch: at arrival for IOC, at the close for expired DAY
        ref = q if tif == "IOC" else last_q
        away_c = round((ref["ask"] - limit) * 100) if o.side == "BUY" else round((limit - ref["bid"]) * 100)
        when = "at arrival" if tif == "IOC" else "at close"
        rows.append({
            "order_id": o.order_id, "time": o.ts.strftime("%H:%M:%S.%f")[:-3],
            "client": o.client_id.replace("CLIENT_", ""), "side": o.side,
            "qty": o.quantity, "filled": int(filled.get(o.order_id, 0)), "limit": limit,
            "bid": q["bid"], "ask": q["ask"], "spread_c": round((q["ask"] - q["bid"]) * 100),
            "principal_pos": pos,
            "reason": ("CANCELLED: " if tif == "IOC" else "EXPIRED: ")
                      + f"non-marketable, {away_c}c away from the touch {when}",
        })
    return pd.DataFrame(rows)


unfilled_ioc = unfilled_table("IOC")
print(f"{len(unfilled_ioc)} cancelled IOC orders, {unfilled_ioc['qty'].sum():,} shares")
unfilled_ioc

In [ ]:
# Expired DAY orders — rested all day, still non-marketable at the final quote
unfilled_day = unfilled_table("DAY")
print(f"{len(unfilled_day)} expired DAY orders, "
      f"{(unfilled_day['qty'] - unfilled_day['filled']).sum():,} shares expired")
unfilled_day